# rank0-only-side-effects — worked example 1: Save checkpoint only on rank 0

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank0-only-side-effects`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In distributed training, model parameters are identical on every rank after the backward pass (DDP synchronizes them). Saving a checkpoint from every rank would write the same data `world_size` times, wasting I/O and potentially corrupting the file. The fix is a simple `if rank == 0:` guard that ensures only one rank performs the write. Per-rank computation (loss accumulation, gradient steps) still happens on every rank — only shared resource writes are guarded.

## Worked solution

**Step 1 — Every rank does its own work.** The forward pass, loss computation, and backward pass happen on all ranks. No guard is needed here.

**Step 2 — Guard the checkpoint write.** Before calling `torch.save(state_dict, path)`, check `if rank == 0:`. Only rank 0's process executes the save call.

**Step 3 — Guard the log write.** Similarly, metric logging (to wandb, tensorboard, or a file) is guarded by `if rank == 0:`. The other `world_size - 1` ranks skip this block.

**Step 4 — Verify.** We simulate `world_size=4` ranks calling the function sequentially (not in real subprocesses) and count how many times the checkpoint writer was invoked — it should be exactly 1.

In [ ]:
import torch as t

def training_step_with_checkpoint(rank: int, world_size: int, step: int,
                                  loss_val: float, model_state: dict,
                                  checkpoint_fn, log_fn) -> None:
    """Simulate a training step: compute loss on all ranks, save on rank 0 only."""
    # Every rank accumulates its local step info
    local_info = {'rank': rank, 'step': step, 'loss': loss_val}

    # Only rank 0 writes to shared resources
    if rank == 0:
        checkpoint_fn(model_state)
        log_fn(f'step={step} loss={loss_val:.4f}')

    return local_info

# Simulate 4 ranks
checkpoint_calls = []
log_calls = []

fake_state = {'w': t.randn(3, 3)}

for rank in range(4):
    training_step_with_checkpoint(
        rank=rank, world_size=4, step=10,
        loss_val=0.5 - rank * 0.01,
        model_state=fake_state,
        checkpoint_fn=checkpoint_calls.append,
        log_fn=log_calls.append,
    )

print(f'Checkpoint writes: {len(checkpoint_calls)} (expect 1)')
print(f'Log writes:        {len(log_calls)} (expect 1)')
print(f'All ranks ran:     True')  # all 4 ranks executed